## Setup Instructions (Google Colab)

### Install Julia (v1.9.3)

In [ ]:
!wget https://julialang-s3.julialang.org/bin/linux/x64/1.9/julia-1.9.3-linux-x86_64.tar.gz
!tar -xvzf julia-1.9.3-linux-x86_64.tar.gz
!sudo mv julia-1.9.3 /opt/julia
!sudo ln -s /opt/julia/bin/julia /usr/local/bin/julia

### Verify Installation

In [ ]:
!julia --version
!jupyter kernelspec list

### Install Required Julia Packages

In [ ]:
!julia -e 'using Pkg; Pkg.add("LibraryAugmentedSymbolicRegression")'
!julia -e 'using Pkg; Pkg.add("MLJ")'
!julia -e 'using Pkg; Pkg.status()'
!julia -e 'using LibraryAugmentedSymbolicRegression; println("Package loaded successfully!")'

### Example Usage (in `script.jl`) - 5D Example

In [ ]:
%%writefile script.jl

using LibraryAugmentedSymbolicRegression
using MLJ

# Dataset with five named features:

# 5D
X = (
    a = rand(500),
    b = rand(500),
    c = rand(500),
    d = rand(500),
    e = rand(500),
)
# and one target:

# 5D
y = @. 2 * cos(X.a * 23.5) - X.b^2 + 0.5 * X.c - 0.3 * X.d^2 + sin(X.e * 3)

# with some noise:
y = y .+ randn(500) .* 1e-3

p = 0.001
model = LaSRRegressor(;
    # SR.jl Options
    niterations=40,
    binary_operators=[+, -, *, /, ^],
    unary_operators=[cos],
    populations=20,
    # LaSR Options
    use_llm=true,
    use_concepts=true,
    use_concept_evolution=true,
    llm_operation_weights=LLMOperationWeights(;
        llm_crossover=p, llm_mutate=p, llm_randomize=p
    ),
    llm_context="We believe the relationship between the theta and offset parameter is a function of the cosine of the theta variable and the square of the offset.",
    variable_names=Dict("a" => "theta", "b" => "offset"),
    prompts_dir="prompts/",
    api_key="token-abc123",
    model="meta-llama/Meta-Llama-3.1-8B-Instruct",
    api_kwargs=Dict("url" => "http://localhost:11440/v1"),
    verbose=true, # Set to true to see LLM generation logs.
)

mach = machine(model, X, y)

# Ensure ./prompts/ exists
fit!(mach)
report(mach)
predict(mach, X)

### Clone and Run Locally

In [ ]:
!git clone https://github.com/trishullab/LibraryAugmentedSymbolicRegression.jl.git
%cd LibraryAugmentedSymbolicRegression.jl

In [ ]:
!julia script.jl